# Preprocess YouTube trending titles

Turns the raw trending parquet into a clean, deduplicated, labeled table ready
for fine-tuning a "viral-worthiness" model on `video_title`.

Steps:
1. Load + cast the string columns to numbers.
2. **Deduplicate by `video_id`** (a video appears once per trending country) and
   keep a `trending_country_count` as its own virality signal.
3. Drop nulls/blanks, clean the title text.
4. Engineer engagement **targets** (like-rate, engagement-rate, log-views) and
   simple title **features**.
5. Build a **viral label** by engagement percentile *within category*.
6. Split train/val/test by `video_id` (no leakage) and save to `processed/`.

Why these targets: every row already trended, so there is no natural "not viral"
class. We derive virality from engagement, preferring **rates** (likes/views)
over raw views, which mostly track channel size rather than the title.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pandas", "pyarrow", "numpy", "scikit-learn"], check=True)

CompletedProcess(args=['/Users/angie/miniforge3/envs/nesso/bin/python', '-m', 'pip', 'install', '-q', 'pandas', 'pyarrow', 'numpy', 'scikit-learn'], returncode=0)

## Config

In [2]:
import os

# Default to the small daily file; set USE_FULL=True for the 10.2M-row global file.
USE_FULL    = False
DAILY_PATH  = "youtube_trending_videos_global_daily.parquet"
FULL_PATH   = "youtube_trending_videos_global.parquet"
# Resolve whether the notebook runs in youtube_title_trending/ or the repo root.
def _find(name):
    for p in [name, os.path.join("youtube_title_trending", name)]:
        if os.path.exists(p):
            return p
    return name
SRC_PATH = _find(FULL_PATH if USE_FULL else DAILY_PATH)
OUT_DIR  = os.path.join(os.path.dirname(SRC_PATH) or ".", "processed")

# Viral label: top quantile of within-category engagement = 1, bottom = 0.
TOP_Q, BOTTOM_Q = 0.75, 0.25
MIN_VIEWS = 1000          # drop ultra-low-view noise
SEED = 42
print("source:", SRC_PATH)

source: youtube_trending_videos_global_daily.parquet


## 1. Load + cast

In [3]:
import pandas as pd
import numpy as np

COLS = ["video_id", "video_title", "video_category_id", "video_trending_country",
        "video_view_count", "video_like_count", "video_comment_count",
        "channel_subscriber_count", "video_published_at"]
df = pd.read_parquet(SRC_PATH, columns=COLS)
print("raw rows:", len(df))

for c in ["video_view_count", "video_like_count", "video_comment_count", "channel_subscriber_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

raw rows: 19181


## 2. Deduplicate by video_id

Collapse the per-country rows into one row per video: take the max engagement
(snapshot timing differs slightly across rows) and count the trending countries.

In [4]:
country_count = df.groupby("video_id")["video_trending_country"].nunique().rename("trending_country_count")

agg = (df.sort_values("video_view_count")
         .groupby("video_id", as_index=False)
         .agg({"video_title": "last",
               "video_category_id": "last",
               "channel_subscriber_count": "max",
               "video_published_at": "last",
               "video_view_count": "max",
               "video_like_count": "max",
               "video_comment_count": "max"}))
agg = agg.merge(country_count, on="video_id")
print("after dedup:", len(agg), "unique videos")

after dedup: 9279 unique videos


## 3. Clean: drop nulls/blanks, tidy titles

In [5]:
before = len(agg)
agg = agg.dropna(subset=["video_title", "video_category_id", "video_view_count",
                         "video_like_count", "video_comment_count"])
# collapse whitespace; drop empties and ultra-low-view rows
agg["video_title"] = agg["video_title"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
agg = agg[(agg["video_title"] != "") & (agg["video_view_count"] >= MIN_VIEWS)]
agg = agg.reset_index(drop=True)
print(f"dropped {before - len(agg)} rows -> {len(agg)} remain")

dropped 125 rows -> 9154 remain


## 4. Targets + title features

In [6]:
import re

v = agg["video_view_count"]
agg["like_rate"]       = agg["video_like_count"] / v
agg["comment_rate"]    = agg["video_comment_count"] / v
agg["engagement_rate"] = (agg["video_like_count"] + agg["video_comment_count"]) / v
agg["log_views"]       = np.log1p(v)

EMOJI = re.compile("[\U0001F000-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]")
t = agg["video_title"]
agg["title_char_len"]   = t.str.len()
agg["title_word_count"] = t.str.split().apply(len)
agg["has_emoji"]        = t.apply(lambda s: bool(EMOJI.search(s)))
agg["has_hashtag"]      = t.str.contains("#")
agg["has_number"]       = t.str.contains(r"\d")
agg["has_question"]     = t.str.contains(r"\?")
agg["has_exclaim"]      = t.str.contains("!")
agg["upper_ratio"]      = t.apply(lambda s: sum(c.isupper() for c in s) / max(len(s), 1))
agg[["video_title", "like_rate", "engagement_rate", "log_views", "trending_country_count"]].head()

,video_title,like_rate,engagement_rate,log_views,trending_country_count
0,🌟Brawl Stars Championship 2026 - Final Mensal ...,0.015818,0.015833,11.069276,2
1,Κρύβεται στο ΔΑΣΟΣ... SIRENHEAD,0.039946,0.042646,10.80501,1
2,DIE POKEGÖTTER mussten uns VERFLUCHEN... bevor...,0.03394,0.036419,10.276843,2
3,FESTIVAL OF FOOTBALL GARETH BALE SQUAD BUILDER...,0.027895,0.029597,10.267714,1
4,Calema - À Prova De Bala,0.020168,0.02119,13.4371,1


## 5. Viral label (engagement percentile within category)

`viral_score` = rank of `engagement_rate` within the video's category (0-1).
`viral_label` = 1 (top quantile) / 0 (bottom quantile); middle rows get NaN so
the binary detector trains on clear cases. The continuous targets stay for
regression/ranking.

In [7]:
agg["viral_score"] = agg.groupby("video_category_id")["engagement_rate"].rank(pct=True)
agg["viral_label"] = np.where(agg["viral_score"] >= TOP_Q, 1,
                       np.where(agg["viral_score"] <= BOTTOM_Q, 0, np.nan))
print(agg["viral_label"].value_counts(dropna=False).to_string())

viral_label
NaN    4574
1.0    2297
0.0    2283


## 6. Split by video_id and save

In [8]:
from sklearn.model_selection import train_test_split

def safe_split(frame, test_size):
    # Stratify by category only when every category has >=2 rows; otherwise a
    # rare single-video category breaks stratification, so fall back to random.
    counts = frame["video_category_id"].value_counts()
    strat = frame["video_category_id"] if counts.min() >= 2 else None
    return train_test_split(frame, test_size=test_size, random_state=SEED, stratify=strat)

train, tmp = safe_split(agg, 0.2)
val, test = safe_split(tmp, 0.5)

os.makedirs(OUT_DIR, exist_ok=True)
for name, part in [("train", train), ("val", val), ("test", test)]:
    path = os.path.join(OUT_DIR, f"{name}.parquet")
    part.reset_index(drop=True).to_parquet(path, index=False)
    print(f"{name}: {len(part):>6} rows -> {path}")

train:   7323 rows -> ./processed/train.parquet
val:    915 rows -> ./processed/val.parquet
test:    916 rows -> ./processed/test.parquet


## 7. Sanity check

In [9]:
print("final columns:\n", list(agg.columns), "\n")
print("most viral titles (engagement_rate, in-category top):")
top = agg.sort_values("viral_score", ascending=False).head(5)
print(top[["video_title", "video_category_id", "engagement_rate", "viral_score"]].to_string(index=False))
print("\nleast viral titles:")
bot = agg.sort_values("viral_score").head(5)
print(bot[["video_title", "video_category_id", "engagement_rate", "viral_score"]].to_string(index=False))

final columns:
 ['video_id', 'video_title', 'video_category_id', 'channel_subscriber_count', 'video_published_at', 'video_view_count', 'video_like_count', 'video_comment_count', 'trending_country_count', 'like_rate', 'comment_rate', 'engagement_rate', 'log_views', 'title_char_len', 'title_word_count', 'has_emoji', 'has_hashtag', 'has_number', 'has_question', 'has_exclaim', 'upper_ratio', 'viral_score', 'viral_label'] 

most viral titles (engagement_rate, in-category top):
                                                         video_title    video_category_id  engagement_rate  viral_score
                         Dispersar | Byakuya Kuchiki (Bleach) | ALBK     Film & Animation         0.230861          1.0
                          Nueva Raza - Lugar Secreto (Video Oficial)       People & Blogs         0.449088          1.0
Mega CROSSTERREIN van mijn PAARDEN MANEGE bouwen! 🤩🐴 *Sims 4 Deel 1*       Pets & Animals         0.072759          1.0
                                           